# 05 · Visualización y comunicación — ENCC 2017

Este cuaderno continúa el análisis de asistencia al cine (`p63`) con un tablero de tres vistas. Define una audiencia y una pregunta, muestra estimaciones ponderadas junto con el tamaño de cada grupo, y exporta una versión interactiva para compartir. La aplicación Streamlit equivalente se encuentra en `src/tablero.py`.

## Objetivo y decisión de diseño

**Audiencia:** equipo que necesita comunicar patrones descriptivos de consumo cultural.

**Pregunta:** ¿cómo varía la asistencia declarada al cine en el último año entre regiones y grupos de edad y sexo?

**Indicador:** porcentaje de respuestas `SI` sobre `SI` + `NO`, ponderado con `pondera_dem`. Se muestran tres cortes complementarios (región, edad y sexo) en una sola pantalla. Las vistas describen asociaciones; no identifican causas.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd()
if not (RAIZ / "proyecto_equipo").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "proyecto_equipo" / "src"))
from visualizaciones_encc import preparar_datos, resumen_ponderado, crear_tablero

ARCHIVO = RAIZ / "proyecto_equipo" / "datos" / "derivados" / "encc_2017_limpio.parquet"
datos = preparar_datos(pd.read_parquet(ARCHIVO))
print(f"Base analítica: {len(datos):,} respuestas válidas · {datos.shape[1]} variables derivadas")
print(f"Período: 2017 | Fuente: ENCC 2017 | Ponderador: pondera_dem")

Base analítica: 2,800 respuestas válidas · 6 variables derivadas
Período: 2017 | Fuente: ENCC 2017 | Ponderador: pondera_dem


## 1. Revisar los grupos

Se informa el `n` sin ponderar además del porcentaje estimado. Así se puede distinguir una categoría con poca muestra de una diferencia respaldada por más observaciones.

In [2]:
tabla_region = resumen_ponderado(datos, "region")
tabla_region[tabla_region.p63 == "SI"][["region", "porcentaje", "n"]].sort_values("porcentaje", ascending=False).round({"porcentaje": 1})

,region,porcentaje,n
1,CABA,53.7,236
11,PATAGONIA,49.2,191
3,CENTRO,45.2,198
5,CUYO,33.0,131
13,PBA,31.0,137
9,NOA,27.8,99
7,NEA,26.2,116


## 2. Tablero

La escala vertical declara que se trata de porcentajes. Los colores se repiten entre vistas para `SI` y `NO`; las etiquetas y el texto emergente también comunican los valores. El detalle emergente incluye el tamaño muestral por respuesta.

In [3]:
figura = crear_tablero(datos)
figura.show()

## 3. Exportar y comunicar

El HTML es interactivo y autónomo: puede abrirse en un navegador sin ejecutar el cuaderno. El archivo se guarda en `proyecto_equipo/salidas/`, directorio de resultados locales que no se versiona.

In [4]:
salida = RAIZ / "proyecto_equipo" / "salidas" / "tablero_encc.html"
salida.parent.mkdir(parents=True, exist_ok=True)
figura.write_html(salida, include_plotlyjs=True, full_html=True)
print(f"Tablero exportado: {salida}")

Tablero exportado: /Users/agus/Desktop/Facultad/CUARTO/Aprendizaje automatico/aa-2026-entorno/proyecto_equipo/salidas/tablero_encc.html


## Lectura, límites y fuente

- El tablero permite comparar estimaciones descriptivas entre los grupos seleccionados; no demuestra que edad, sexo o región causen la asistencia.
- El tamaño muestral es el número de registros válidos de cada grupo. `pondera_dem` modifica la estimación porcentual, no el `n` informado.
- No se agregan intervalos de confianza: este análisis no dispone de la información completa del diseño muestral necesaria para estimarlos de forma defendible.
- La asistencia corresponde a la respuesta declarada para el último año (`p63`) en el relevamiento 2017. Los resultados no describen necesariamente la situación actual.
- **Fuente:** Secretaría de Cultura de la Nación, Encuesta Nacional de Consumos Culturales 2017, datos.gob.ar. La base analítica proviene del parquet generado en el cuaderno 02.

## Aplicación interactiva

Desde la raíz del repositorio, con las dependencias instaladas:

```bash
streamlit run proyecto_equipo/src/tablero.py
```

Los filtros de región, sexo y edad se aplican a los indicadores y a las tres vistas al mismo tiempo.